In [ ]:
!pip install fastdtw pandas numpy scipy scikit-learn

In [ ]:
import pandas as pd
import numpy as np
from fastdtw import fastdtw
from scipy.spatial.distance import euclidean
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import warnings

warnings.filterwarnings('ignore')

**1. BỘ LỌC TOÁN HỌC (PRE-FILTER) LỌC VIDEO ĐỨNG YÊN**

In [ ]:
# [CẢI TIẾN 1]: Xây dựng bộ lọc đo lường biên độ dao động.
# Thuật toán sẽ tính phương sai (Variance) chuyển động của Cổ tay và Cổ chân.
# Nếu người đứng yên, tọa độ xê dịch cực thấp -> Đánh FAIL 100% ngay từ đầu, không đưa vào AI.
def is_dancing(df_student, variance_threshold=0.005):
    key_joints = ['left_wrist', 'right_wrist', 'left_ankle', 'right_ankle']
    df_key = df_student[df_student['Joint_Name'].isin(key_joints)]
    
    # Tính độ xê dịch (std) trung bình của các khớp ngọn
    movement_std = df_key.groupby('Joint_Name')[['X', 'Y']].std().mean().mean()
    
    if movement_std < variance_threshold:
        return False, movement_std
    return True, movement_std

**2. TIỀN XỬ LÝ & TẠO LABEL THEO TRỌNG SỐ (WEIGHTED JOINTS)**

In [ ]:
def min_max_scale(series):
    return (series - series.min()) / (series.max() - series.min() + 1e-9)

def preprocess_data(df):
    df = df.sort_values(by=['Joint_Name', 'Frame']).reset_index(drop=True)
    mask = df['Visibility'] < 0.5
    df.loc[mask, ['X', 'Y', 'Z']] = np.nan
    df[['X', 'Y', 'Z']] = df.groupby('Joint_Name')[['X', 'Y', 'Z']].transform(
        lambda x: x.interpolate(method='linear', limit_direction='both')
    )
    df.fillna(0, inplace=True) 
    df['X_norm'] = min_max_scale(df['X'])
    df['Y_norm'] = min_max_scale(df['Y'])
    df['Z_norm'] = min_max_scale(df['Z'])
    return df.sort_values(by=['Frame', 'Joint_Name']).reset_index(drop=True)

def build_frame_level_dataset(df_student, df_teacher):
    def get_vectors(df):
        df_unique = df.drop_duplicates(subset=['Frame', 'Joint_Name'])
        wide = df_unique.pivot(index='Frame', columns='Joint_Name', values=['X_norm', 'Y_norm', 'Z_norm'])
        wide.fillna(0, inplace=True)
        return wide.values, wide.index.tolist()

    vec_s, frames_s = get_vectors(df_student)
    vec_t, frames_t = get_vectors(df_teacher)
    distance, path = fastdtw(vec_s, vec_t, dist=euclidean)
    
    aligned_pairs = [{'Frame_s': frames_s[s_idx], 'Frame_t': frames_t[t_idx]} for s_idx, t_idx in path]
    df_path = pd.DataFrame(aligned_pairs)
    
    df_s_aligned = pd.merge(df_path, df_student, left_on='Frame_s', right_on='Frame').drop(columns=['Frame'])
    df_t_aligned = pd.merge(df_path, df_teacher, left_on='Frame_t', right_on='Frame').drop(columns=['Frame'])
    
    df_s_aligned = df_s_aligned.rename(columns={'X_norm': 'X_s', 'Y_norm': 'Y_s', 'Z_norm': 'Z_s', 'Label': 'Label_s'})
    df_t_aligned = df_t_aligned.rename(columns={'X_norm': 'X_t', 'Y_norm': 'Y_t', 'Z_norm': 'Z_t'})
    
    label_col = 'Label_s' if 'Label_s' in df_s_aligned.columns else 'Label'
    
    df_merged = pd.merge(
        df_s_aligned[['Frame_s', 'Frame_t', 'Joint_Name', 'X_s', 'Y_s', 'Z_s', label_col]], 
        df_t_aligned[['Frame_s', 'Frame_t', 'Joint_Name', 'X_t', 'Y_t', 'Z_t']], 
        on=['Frame_s', 'Frame_t', 'Joint_Name']
    )
    
    # Tính Error & Cosine
    df_merged['Joint_Error'] = np.sqrt((df_merged['X_s'] - df_merged['X_t'])**2 + (df_merged['Y_s'] - df_merged['Y_t'])**2 + (df_merged['Z_s'] - df_merged['Z_t'])**2)
    dot = (df_merged['X_s'] * df_merged['X_t']) + (df_merged['Y_s'] * df_merged['Y_t']) + (df_merged['Z_s'] * df_merged['Z_t'])
    norm_s = np.sqrt(df_merged['X_s']**2 + df_merged['Y_s']**2 + df_merged['Z_s']**2)
    norm_t = np.sqrt(df_merged['X_t']**2 + df_merged['Y_t']**2 + df_merged['Z_t']**2)
    df_merged['Joint_Cosine'] = dot / (norm_s * norm_t + 1e-9)
    
    # Xoay ngang thành Frame-level
    wide_error = df_merged.pivot_table(index='Frame_s', columns='Joint_Name', values='Joint_Error', fill_value=0)
    wide_error.columns = [f"Error_{c}" for c in wide_error.columns]
    
    wide_cosine = df_merged.pivot_table(index='Frame_s', columns='Joint_Name', values='Joint_Cosine', fill_value=0)
    wide_cosine.columns = [f"Cosine_{c}" for c in wide_cosine.columns]
    
    wide_label = df_merged.pivot_table(index='Frame_s', columns='Joint_Name', values=label_col, fill_value=0)
    
    # =========================================================================
    # [CẢI TIẾN 2]: ĐÁNH TRỌNG SỐ CHO CÁC KHỚP (ÉP MÔ HÌNH NHÌN VÀO TAY CHÂN)
    # =========================================================================
    # Mũi, Vai, Hông: Khớp trụ (Trọng số 1.0)
    # Khuỷu tay, Đầu gối: Khớp giữa (Trọng số 1.5)
    # Cổ tay, Cổ chân: Khớp ngọn - Quyết định đúng/sai bước nhảy (Trọng số 2.0)
    # TỔNG ĐIỂM TỐI ĐA: 19.0 điểm.
    
    weights = {
        'nose': 1.0, 'left_shoulder': 1.0, 'right_shoulder': 1.0, 'left_hip': 1.0, 'right_hip': 1.0,
        'left_elbow': 1.5, 'right_elbow': 1.5, 'left_knee': 1.5, 'right_knee': 1.5,
        'left_wrist': 2.0, 'right_wrist': 2.0, 'left_ankle': 2.0, 'right_ankle': 2.0
    }
    
    total_score = pd.Series(0.0, index=wide_label.index)
    for joint, weight in weights.items():
        if joint in wide_label.columns:
            total_score += wide_label[joint] * weight
            
    # [CẢI TIẾN 3]: NÂNG BAR ĐÁNH GIÁ (Khắt khe hơn)
    # Để Pass, Frame phải đạt tối thiểu 12.5 / 19.0 điểm.
    # Nếu chỉ đúng mặt và vai (5 điểm), học sinh sẽ FAIL chắc chắn. 
    # Bắt buộc phải đúng cử động cổ tay/cổ chân mới đủ điểm vớt thành PASS.
    frame_pass_threshold = 12.5 
    frame_label = (total_score >= frame_pass_threshold).astype(int)
    
    frame_dataset = pd.concat([wide_error, wide_cosine], axis=1).reset_index()
    frame_dataset['Frame_Label'] = frame_label.values
    
    return frame_dataset

**3. TẠO DATASET & CHẶN VIDEO RÁC**

In [ ]:
dance_pairs = [
    {"id": 1, "student": "student_dance_1.csv", "teacher": "teacher_dance_1.csv"},
    {"id": 2, "student": "student_dance_2.csv", "teacher": "teacher_dance_2.csv"},
    {"id": 3, "student": "student_dance_3.csv", "teacher": "teacher_dance_3.csv"},
    {"id": 4, "student": "student_dance_4.csv", "teacher": "teacher_dance_4.csv"},
    {"id": 5, "student": "student_dance_5.csv", "teacher": "teacher_dance_5.csv"},
    {"id": 6, "student": "student_dance_bad.csv", "teacher": "teacher_dance_bad.csv"}, # Video này đứng im
    {"id": 7, "student": "student_dance_stand.csv", "teacher": "teacher_dance_stand.csv"}
]

TEST_VIDEO_IDS = [5, 6]  # Tập Test
all_frames = []

print("=== BƯỚC 1: QUÉT & TRÍCH XUẤT DỮ LIỆU ===")
for pair in dance_pairs:
    try:
        df_s = pd.read_csv(pair["student"])
        df_t = pd.read_csv(pair["teacher"])
        
        # Áp dụng bộ lọc Toán học
        is_moving, std_dev = is_dancing(df_s)
        
        if not is_moving:
            print(f"[X] Video {pair['id']} bị chặn (Std={std_dev:.4f}): Phát hiện người học đứng yên!")
            if pair['id'] in TEST_VIDEO_IDS:
                print(f"   -> KẾT QUẢ: Video {pair['id']} bị hệ thống tự động chấm 0 điểm (FAIL) không cần AI.")
            # Bỏ qua không nạp vào mảng Train để bảo vệ DTW khỏi dữ liệu rác
            continue
            
        print(f"[V] Video {pair['id']} hợp lệ (Std={std_dev:.4f}): Đang xử lý không gian...")
        df_s_clean, df_t_clean = preprocess_data(df_s), preprocess_data(df_t)
        frame_df = build_frame_level_dataset(df_s_clean, df_t_clean)
        frame_df['Video_ID'] = pair['id']
        
        all_frames.append(frame_df)
    except Exception as e:
        print(f"[-] Lỗi tại Video {pair['id']}: {e}")

master_dataset = pd.concat(all_frames, ignore_index=True)
print(f"\n-> Tổng số Frame hợp lệ thu thập được: {len(master_dataset)}")

**4. HUẤN LUYỆN RANDOM FOREST & PHÂN TÍCH TRỌNG SỐ ĐẶC TRƯNG**

In [ ]:
train_data = master_dataset[~master_dataset['Video_ID'].isin(TEST_VIDEO_IDS)]
test_data = master_dataset[master_dataset['Video_ID'].isin(TEST_VIDEO_IDS)]

feature_cols = [col for col in master_dataset.columns if col.startswith('Error_') or col.startswith('Cosine_')]
X_train = train_data[feature_cols]
y_train = train_data['Frame_Label']
X_test = test_data[feature_cols]
y_test = test_data['Frame_Label']

print(f"Dữ liệu Train AI: {X_train.shape[0]} frames")
print(f"Dữ liệu Test AI : {X_test.shape[0]} frames")

print("\nĐang huấn luyện mô hình Random Forest...")
rf_model = RandomForestClassifier(n_estimators=100, max_depth=15, min_samples_leaf=5, random_state=42)
rf_model.fit(X_train, y_train)

if len(X_test) > 0:
    y_pred = rf_model.predict(X_test)

    print("\n======================================================")
    print(f"   KẾT QUẢ ĐÁNH GIÁ CẤP ĐỘ FRAME (TEST VIDEO)")
    print("======================================================\n")
    print(classification_report(y_test, y_pred, target_names=['Fail (0)', 'Pass (1)'] if len(set(y_test)) > 1 else None))
    
    acc = accuracy_score(y_test, y_pred)
    print(f"-> ĐỘ CHÍNH XÁC TỔNG THỂ (Accuracy): {acc*100:.2f}%")

    cm = confusion_matrix(y_test, y_pred)
    print(f"\nCHI TIẾT TRÊN TỔNG SỐ {len(y_test)} FRAMES TEST:")
    if cm.shape == (2,2):
        print(f"- Số Frames FAIL thực, máy chấm chuẩn FAIL : {cm[0][0]}")
        print(f"- Số Frames FAIL thực, máy chấm lọt PASS   : {cm[0][1]} (Báo động giả)")
        print(f"- Số Frames PASS thực, máy chấm oan FAIL   : {cm[1][0]} (Đánh hỏng oan)")
        print(f"- Số Frames PASS thực, máy chấm chuẩn PASS : {cm[1][1]}")
    
# Kiểm tra xem Mô hình đã thoát cảnh "Mù chân" chưa
importances = rf_model.feature_importances_
feature_imp_df = pd.DataFrame({'Đặc trưng': feature_cols, 'Mức độ đóng góp (%)': importances * 100})
print("\n=== TOP 10 ĐẶC TRƯNG QUAN TRỌNG (HÃY KIỂM TRA XEM CÓ ANKLE/KNEE KHÔNG) ===")
display(feature_imp_df.sort_values(by='Mức độ đóng góp (%)', ascending=False).head(10))